# Drominator — Entraînement sur Google Colab

Pipeline complet : **Panda3D** → **YOLO** → **Depth Anything V2** → **PPO (stable-baselines3)**

> Assure-toi d'utiliser un **Runtime GPU** : `Exécution > Modifier le type d'exécution > T4 GPU`

## 0 — Vérification du GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Pas de GPU détecté ! Active le runtime GPU dans Exécution > Modifier le type d'exécution.")

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 — Installation des dépendances système

In [ ]:
%%bash
apt-get install -y -q \
    libgl1-mesa-glx \
    libgles2-mesa \
    libgl1-mesa-dri \
    libegl1-mesa \
    xvfb 2>&1 | tail -5
echo "Deps système OK"

## 2 — Installation des dépendances Python

In [ ]:
%%bash
pip install -q \
    panda3d==1.10.14 \
    stable-baselines3[extra] \
    gymnasium \
    ultralytics \
    transformers \
    accelerate \
    opencv-python-headless \
    Pillow \
    numpy \
    tensorboard 2>&1 | tail -5
echo "Deps Python OK"

## 3 — Clonage du dépôt

Le dépôt est privé sur GitHub. Deux options :

**Option A** — Token GitHub (recommandé pour Colab)  
Génère un token sur https://github.com/settings/tokens (scope `repo`), colle-le dans la variable ci-dessous.

**Option B** — Monte Google Drive et copie le dossier manuellement.

In [ ]:
import os

# ── Option A : GitHub token ────────────────────────────────────────────────────
GITHUB_TOKEN = ""   # <-- colle ton token ici  (ex: ghp_xxxxxxxxxxxx)
REPO_URL     = "https://github.com/matensu/Drominator.git"
PROJECT_DIR  = "/content/Drominator"

if GITHUB_TOKEN:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    if not os.path.exists(PROJECT_DIR):
        os.system(f"git clone {auth_url} {PROJECT_DIR}")
    else:
        print("Dossier déjà présent, pull…")
        os.system(f"git -C {PROJECT_DIR} pull")
else:
    print("GITHUB_TOKEN vide — utilise l'Option B (Drive) ou remplis le token.")

# ── Option B : Google Drive ───────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = "/content/drive/MyDrive/Drominator"   # adapter le chemin

print(f"Projet : {PROJECT_DIR}")
os.listdir(PROJECT_DIR)

## 4 — Téléchargement du modèle YOLO

`yolo11n.pt` est dans le `.gitignore`. Ultralytics le télécharge automatiquement.

In [ ]:
import os
from ultralytics import YOLO

YOLO_PATH = os.path.join(PROJECT_DIR, "yolo11n.pt")

if not os.path.exists(YOLO_PATH):
    print("Téléchargement yolo11n.pt…")
    # YOLO() télécharge dans le cwd → on se place dans le projet
    os.chdir(PROJECT_DIR)
    model = YOLO("yolo11n.pt")   # auto-download
    # Si téléchargé dans le dossier courant de l'utilisateur, on le déplace
    downloaded = os.path.expanduser("~/.config/Ultralytics/Assets/yolo11n.pt")
    if os.path.exists(downloaded) and not os.path.exists(YOLO_PATH):
        import shutil
        shutil.copy(downloaded, YOLO_PATH)
    print("OK")
else:
    print(f"yolo11n.pt déjà présent ({os.path.getsize(YOLO_PATH)/1e6:.1f} MB)")

## 5 — (Optionnel) Monter Drive pour sauvegarder les checkpoints

In [ ]:
# Décommente pour activer la sauvegarde automatique sur Drive

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_MODELS = "/content/drive/MyDrive/Drominator_models"
# os.makedirs(DRIVE_MODELS, exist_ok=True)

# # Symlink pour que stable-baselines3 sauvegarde directement sur Drive
# LOCAL_MODELS = os.path.join(PROJECT_DIR, "ai", "models_pipeline")
# if not os.path.islink(LOCAL_MODELS) and not os.path.exists(LOCAL_MODELS):
#     os.symlink(DRIVE_MODELS, LOCAL_MODELS)

print("(Drive non monté — les checkpoints seront dans /content/Drominator/ai/models_pipeline/)")

## 6 — Lancement de l'entraînement

Le script tourne en **subprocess** pour respecter le chargement Panda3D en tête de module.  
Les logs s'affichent en temps réel.  
Pour arrêter proprement : **Exécution > Interrompre l'exécution** (équivalent Ctrl+C — le modèle est sauvegardé).

In [ ]:
import subprocess, sys, os

os.chdir(PROJECT_DIR)

env = os.environ.copy()
# Panda3D offscreen sur serveur headless
env["DISPLAY"] = ":99"

# Lance un serveur X virtuel en arrière-plan (au cas où Panda3D en aurait besoin)
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1x1x24"])

print("=" * 60)
print("Démarrage de l'entraînement…")
print("Ctrl+C / Interrompre = sauvegarde propre")
print("=" * 60)

proc = subprocess.Popen(
    [sys.executable, "-m", "ai.train_full_pipeline"],
    cwd=PROJECT_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.send_signal(__import__('signal').SIGINT)
    proc.wait()
    print("\nEntraînement interrompu — modèle sauvegardé.")

proc.wait()
print(f"\nCode de sortie : {proc.returncode}")

## 7 — TensorBoard (visualisation des courbes)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/Drominator/ai/logs_pipeline

## 8 — Télécharger le modèle final

In [ ]:
import os
from google.colab import files

MODEL_PATH = "/content/Drominator/ai/models_pipeline/drone_pipeline_final.zip"

if os.path.exists(MODEL_PATH):
    files.download(MODEL_PATH)
else:
    # Liste les checkpoints disponibles
    models_dir = "/content/Drominator/ai/models_pipeline"
    ckpts = sorted(os.listdir(models_dir)) if os.path.exists(models_dir) else []
    if ckpts:
        latest = os.path.join(models_dir, ckpts[-1])
        print(f"Modèle final non trouvé. Dernier checkpoint : {latest}")
        files.download(latest)
    else:
        print("Aucun modèle trouvé. Lance d'abord l'entraînement (cellule 6).")